# Handel tournament analysis

Explore a tournament written by `handel-tournament`: who wins, how they negotiate, and whether
their trades were good by the Monte Carlo fair value (Phase 4).

1. Set `RESULTS` below to a tournament directory (for example `results/tournament_cheap`).
2. Run `uv run handel-analyze <dir> --value` once first: it values every offer with bot rollouts
   (free, no API calls). Without it the trade-quality sections stay empty.
3. Run all cells (`uv sync --group analysis` installs matplotlib and Jupyter).

Intervals are 95%. With fewer than 5 seeds they are Wilson intervals that treat games as
independent (marked `w`), which is too optimistic; read small tournaments as descriptions of
those games, not of the models.

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

from handel import analysis, charts

ROOT = Path.cwd() if (Path.cwd() / "results").exists() else Path.cwd().parent
CANDIDATES = ["results/tournament_cheap", "results/tournament_claude", "results/tournament_offline"]
RESULTS = next((ROOT / c for c in CANDIDATES if (ROOT / c / "games.jsonl").exists()), ROOT / CANDIDATES[0])

data = analysis.load(RESULTS)
OUT = RESULTS / "analysis"
OUT.mkdir(parents=True, exist_ok=True)
seeds = len({x["seed"] for x in data.lines})
print(f"{RESULTS.name}: {len(data.lines)} games, {seeds} seeds, entrants {data.entrants}")
print(f"cost ${sum(x['cost_usd'] for x in data.lines):.2f}; "
      f"{sum(data.value(x) is not None for x in data.lines)} games have fair-value reports")

## Who wins

Win rate with its interval, then average placement and the head-to-head matrix. With four
seats a fair share is 25%; every seed is played in every seat rotation, so seat luck cancels
within a seed.

In [ ]:
display(Image(charts.win_rate_chart(data, OUT / "win_rates.png")))
rows = analysis.outcome_table(data)
display(Markdown("\n".join(analysis._table(
    ["entrant", "games", "win rate", "avg place", "avg net worth"],
    [[r["entrant"], str(r["games"]), r["win_rate"].fmt(), r["placement"].fmt(pct=False),
      f"{r['net_worth'].value or 0:,.0f}"] for r in rows]))))

In [ ]:
p = charts.head_to_head_chart(data, OUT / "head_to_head.png")
if p:
    display(Image(p))

## How they negotiate

Counts from the negotiation threads: proposals, how offers were answered, deals and promises.

In [ ]:
neg = analysis.negotiation_table(data)
display(Markdown("\n".join(analysis._table(
    ["entrant", "proposals / game", "proposals that became deals", "offers faced", "accepted",
     "countered", "rejected", "deals", "promises"],
    [[r["entrant"], r["proposals_per_game"].fmt(pct=False), r["proposal_deal_rate"].fmt(),
      str(r["offers_faced"]), r["accept_rate"].fmt(), r["counter_rate"].fmt(), r["reject_rate"].fmt(),
      str(r["deals"]), str(r["promises"])] for r in neg]))))

## Were the trades good?

Each dot is an offer a player had to answer, placed at the change in *that player's* win
probability if they accepted (Monte Carlo, heuristic-bot rollouts). Filled = accepted,
hollow = rejected or countered. Good play fills the right side and leaves the left hollow.

In [ ]:
p = charts.offers_chart(data, OUT / "offers.png")
if p:
    display(Image(p))
    tq = analysis.trade_quality_table(data)
    display(Markdown("\n".join(analysis._table(
        ["entrant", "offers answered", "decision accuracy", "deals", "win prob. gained per deal (pts)",
         "edge over partner (pts)", "price vs fair (fl)"],
        [[r["entrant"], str(r["offers_valued"]), r["decision_accuracy"].fmt(digits=0), str(r["deals"]),
          analysis._pts(r["deal_gain"]), analysis._pts(r["deal_edge"]),
          "-" if r["price_edge_fl"].value is None else f"{r['price_edge_fl'].value:+.0f}"] for r in tq]))))
else:
    print("No fair-value reports yet: run `uv run handel-analyze", RESULTS, "--value` first.")

### The worst accepted offers

Deals where the player who accepted lost the most win probability, with the chat that led to them.

In [ ]:
from handel.llm.observation import describe_offer
from handel.replay import offer_from_dict

js = analysis.judgements(data)
worst = sorted((j for j in js if j.response == "accept"), key=lambda j: j.d_responder)[:5]
for j in worst:
    line = next(x for x in data.lines if x["game_id"] == j.game_id)
    rec = data.record(line)
    neg = next(n for n in rec["negotiations"] if n["tid"] == j.tid)
    names = line["lineup"]
    text = [f"**{j.game_id}, round {j.round + 1}**: {j.responder} accepted from {j.proposer} "
            f"({j.d_responder * 100:+.1f} pts for {j.responder}, {j.d_proposer * 100:+.1f} for {j.proposer})"]
    for st in neg["steps"]:
        o = st["offer"]
        terms = describe_offer(offer_from_dict(o)).replace(f"P{o['proposer']}", names[o["proposer"]]).replace(
            f"P{o['responder']}", names[o["responder"]])
        note = f' "{o["message"]}"' if o.get("message") else ""
        text.append(f"- {names[st['p']]} ({st['kind']}): {terms}.{note}")
    if neg["reply"]:
        text.append(f"- {names[neg['reply']['p']]} ({neg['reply']['kind']}): \"{neg['reply']['message']}\"")
    display(Markdown("\n".join(text)))

## One game, round by round

Win probability through the first game (by seed and rotation) that has a fair-value timeline.

In [ ]:
p = charts.timeline_chart(data, OUT / "timeline.png")
if p:
    display(Image(p))

## How many more games?

Games needed per entrant for a given 95% margin, assuming independent games (rotations of a
seed are partly correlated, so the real need is somewhat higher).

In [ ]:
fair = 1 / data.seats
for m in (0.15, 0.10, 0.05):
    print(f"±{m * 100:.0f} points: {analysis.games_needed(fair, m)} games per entrant at a "
          f"{fair:.0%} win rate, {analysis.games_needed(0.5, m)} for a 50/50 head-to-head")